# 第 11 章: 評価指標の探索と可視化

Survived の決定木について、混同行列と ROC 曲線でモデルの当たり方を確認する。

In [ ]:
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from japanese_font import use_japanese_font
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.tree import DecisionTreeClassifier

from lib.chapter02.iris_preprocessing import split_train_test
from lib.chapter11.datasets import prepare_survived
from lib.chapter11.evaluation import confusion_matrix, f1_score, precision, recall
from lib.dataset import data_dir

use_japanese_font();

In [ ]:
x, t = prepare_survived(pd.read_csv(data_dir() / "Survived.csv"))
split = split_train_test(x, t, test_size=0.3, seed=0)
model = DecisionTreeClassifier(max_depth=2, random_state=0)
model.fit(split.x_train, split.t_train)
predicted = model.predict(split.x_test)
cm = confusion_matrix(split.t_test, predicted, positive=1)
cm

In [ ]:
sns.heatmap(
    [[cm.tn, cm.fp], [cm.fn, cm.tp]],
    annot=True,
    fmt="d",
    xticklabels=["死亡と予測", "生存と予測"],
    yticklabels=["実際は死亡", "実際は生存"],
)
plt.title("混同行列（テストデータ）");

In [ ]:
{
    "適合率": round(precision(cm), 4),
    "再現率": round(recall(cm), 4),
    "F値": round(f1_score(cm), 4),
}

In [ ]:
probability = model.predict_proba(split.x_test)[:, 1]
fpr, tpr, _ = roc_curve(split.t_test, probability)
plt.plot(fpr, tpr, label="決定木（max_depth=2）")
plt.plot([0, 1], [0, 1], linestyle="--", label="でたらめな予測")
plt.xlabel("偽陽性率")
plt.ylabel("真陽性率（再現率）")
plt.title("ROC 曲線")
plt.legend();

In [ ]:
{
    depth: round(
        roc_auc_score(
            split.t_test,
            DecisionTreeClassifier(max_depth=depth, random_state=0)
            .fit(split.x_train, split.t_train)
            .predict_proba(split.x_test)[:, 1],
        ),
        4,
    )
    for depth in [1, 2, 4, 8]
}